### Applied Deep Learning Project

##### Name: Sayed Saidoo

##### Dataset: DailyDialog

##### Project Overview

This project applies deep learning techniques to the DailyDialog dataset to develop a decoder-only Transformer capable of generating coherent conversational text through autoregressive language modeling. The dataset contains human-written, multi-turn dialogues covering a variety of everyday topics and provides a suitable foundation for learning conversational structure, contextual language patterns, and next-token prediction.

The primary objective of this project is to design, train, and evaluate a GPT-style Transformer model using PyTorch while following a complete deep learning experimentation workflow. The analysis includes dataset inspection, preprocessing, tokenization, model implementation, training, experimental comparison, performance evaluation, and interpretation of generated text outputs.

The study focuses on the following research question:

*Can a compact decoder-only Transformer learn meaningful conversational patterns from human-authored dialogue and generate coherent text continuations while operating within modest computational constraints?*

To address this question, a baseline Transformer model will be trained on the DailyDialog dataset and evaluated using appropriate language modeling metrics and qualitative text generation examples. A controlled comparison experiment will be conducted to investigate the impact of an architectural or hyperparameter modification on model performance.

##### Setup Core Dependencies

In [2]:
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer
from typing import Any
from pathlib import Path

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

PyTorch: 2.14.0+cpu
CUDA Available: False


In [4]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

print(tokenizer.vocab_size)

50257


### Data Ingestion and Inspection

In [8]:
# create path variables for ease of navigation
DATA_PATH = Path("../data/raw/train")

dialogues_path = DATA_PATH / "dialogues_train.txt"
emotion_path = DATA_PATH / "dialogues_emotion_train.txt"
act_path = DATA_PATH / "dialogues_act_train.txt"


In [9]:
with open(dialogues_path, "r", encoding="utf-8") as f:
    dialogues = f.readlines()

with open(emotion_path, "r", encoding="utf-8") as f:
    emotions = f.readlines()

with open(act_path, "r", encoding="utf-8") as f:
    acts = f.readlines()

print(f"Dialogues: {len(dialogues):,}")
print(f"Emotion Labels: {len(emotions):,}")
print(f"Dialogue Acts: {len(acts):,}")

Dialogues: 11,118
Emotion Labels: 11,118
Dialogue Acts: 11,118


In [10]:
assert len(dialogues) == len(emotions) == len(acts)

print("Dataset alignment verified.")

Dataset alignment verified.


##### Dataset Components

The DailyDialog dataset consists of three aligned files:

- `dialogues_train.txt` containing the dialogue text.
- `dialogues_emotion_train.txt` containing emotion annotations.
- `dialogues_act_train.txt` containing dialogue act annotations.

For the purposes of autoregressive language modeling, the dialogue text will serve as the primary training corpus. Emotion and dialogue act annotations will be inspected during exploration but are not used during baseline model training. These annotations represent potential extensions for emotion-aware or conditional language generation (which can be pretty cool as a next project).

In [11]:
print(dialogues[0])
print(emotions[0])
print(acts[0])

Say , Jim , how about going for a few beers after dinner ? __eou__ You know that is tempting but is really not good for our fitness . __eou__ What do you mean ? It will help us to relax . __eou__ Do you really think so ? I don't . It will just make us fat and act silly . Remember last time ? __eou__ I guess you are right.But what shall we do ? I don't feel like sitting at home . __eou__ I suggest a walk over to the gym where we can play singsong and meet some of our friends . __eou__ That's a good idea . I hear Mary and Sally often go there to play pingpong.Perhaps we can make a foursome with them . __eou__ Sounds great to me ! If they are willing , we could ask them to go dancing with us.That is excellent exercise and fun , too . __eou__ Good.Let ' s go now . __eou__ All right . __eou__

0 0 0 0 0 0 4 4 4 4 

3 4 2 2 2 3 4 1 3 4 

